<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_4_5_MLR_Ames_Part5_Revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLR Predicting Housing Prices in Ames Iowa: Part 5
## Beyond Lines — Tree-Based Methods

**Summary:** In Parts 1-4, we focused on linear models (OLS, Ridge, Lasso, ElasticNet). These are powerful and interpretable but they assume the relationship between features and house prices is a straight line. In this notebook, we explore **Tree-Based Ensembles** (Random Forests and Gradient Boosting) which can automatically capture complex, non-linear patterns and feature interactions.

**Data Source:** http://jse.amstat.org/v19n3/decock/AmesHousing.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Data source
url = 'https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_housing_ames.txt'

# Loading the dataframe
df_raw = pd.read_csv(url, sep='\t')

# Initial cleaning: remove extreme outliers
df = df_raw.loc[df_raw['Gr Liv Area'] < 4000, :].copy()

# Fix Garage Year Built
df.loc[df['Garage Yr Blt'] > 2010, 'Garage Yr Blt'] = 2010

# helper function
def safe_drop(df: pd.DataFrame, drop_list: list[str]) -> pd.DataFrame:
  existing_cols_to_drop = [col for col in drop_list if col in df.columns]
  if existing_cols_to_drop:
    df = df.drop(existing_cols_to_drop, axis=1)
  return df

# compressed cleaning code from previous parts
if 'garage_attached' not in df.columns:
  df['garage_attached'] = np.where(df['Garage Type'] == 'Attchd', 1, 0)
if 'garage_unfinished' not in df.columns:
  df['garage_unfinished'] = np.where(df['Garage Finish'] != 'Unf', 1, 0)

d1 = ['Order', 'PID']
d2 = ['Pool QC', 'Pool Area', 'Misc Feature', 'Misc Val']
d3 = ['Alley', 'Fence', 'Mas Vnr Type', 'Bsmt Qual', 'Bsmt Cond', \
      'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Fireplace Qu', \
      'Neighborhood', 'MS Subclass', 'Mo Sold', 'Kitchen Qual', 'Exter Qual', \
      'Heating QC']
d4 = ['Garage Qual', 'Garage Cond', 'Garage Type', 'Garage Finish']
d5 = ['Street', 'Land Contour', 'Utilities', \
      'Land Slope', 'Condition 1', 'Condition 2',\
      'Roof Matl', 'Exter Cond', 'Heating', 'Central Air', \
      'Electrical', 'Functional', 'Paved Drive', "Sale Type"]

df = safe_drop(df, d1 + d2 + d3 + d4 + d5)

for col in df.select_dtypes(include=['object']).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.50:
    top_value = (df[col].value_counts(normalize=True, dropna=False).index[0])
    df[col + '_' + top_value] = np.where(df[col] == top_value, 1, 0)
    df = safe_drop(df, [col])

df.loc[~df['Foundation'].isin(['PConc', 'CBlock']), 'Foundation'] = 'Other'

df = safe_drop(df, ['Exterior 1st', 'Exterior 2nd'])

for col in df.select_dtypes('object').columns:
  df[col] = df[col].astype('category')

for col in df.columns:
  if df[col].value_counts().shape[0] == 2:
    df[col] = df[col].astype('bool')

for col in df.select_dtypes(include=np.number).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.90:
    df = safe_drop(df, [col])

for col in df.select_dtypes(include=np.number).columns:
  if df[col].value_counts(normalize=True, dropna=False).max() > 0.80:
    if len(df[col].unique()) > 2:
      df[col] = np.where(df[col] > 0, 1, 0)
    df[col] = df[col].astype('boolean')

df = safe_drop(df, ['Mas Vnr Area'])

num_cols_with_na = df.select_dtypes(include=np.number).columns[df.select_dtypes(include=np.number).isnull().any()].tolist()
for col in num_cols_with_na:
    df[col] = df[col].fillna(df[col].median())

df = pd.get_dummies(df, columns=df.select_dtypes(include='category').columns, drop_first=True)

from sklearn.model_selection import train_test_split
X = df.drop(columns=['SalePrice'])
y = df['SalePrice'].map(np.log)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Setup complete. Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## 1. Why Tree-Based Methods?

Linear regression models calculate a single global slope for each feature. But real-world data is often "non-linear."

For example:
*   **Diminishing Returns**: Increasing `Garage Area` from 0 to 400 sq ft adds a lot of value. Increasing it from 1000 to 1400 adds much less.
*   **Threshold Effects**: Houses built *after* a certain year might command a specific premium due to building codes or styles.
*   **Interactions**: `Overall Qual` might matter more in certain neighborhoods than others.

Trees handle these scenarios automatically by splitting the data into subsets.

## 2. Decision Tree Regressor

A Decision Tree works like a flowchart. At each node, it asks a question (e.g., "Is Gr Liv Area > 1500?") and moves to the next branch based on the answer.

**The Danger: Overfitting**
If we don't limit the depth of the tree, it will keep splitting until every house in the training set has its own leaf node. This leads to perfect training accuracy but terrible predictions on new data.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

# 1. Unconstrained Tree (Overfits)
dt_overfit = DecisionTreeRegressor(random_state=42)
dt_overfit.fit(X_train, y_train)

# 2. Constrained Tree (Generalizes better)
dt_limited = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_limited.fit(X_train, y_train)

print(f"Unconstrained Tree Training R²: {dt_overfit.score(X_train, y_train):.4f}")
print(f"Unconstrained Tree Test R²:     {dt_overfit.score(X_test, y_test):.4f}")
print("-" * 30)
print(f"Limited Tree Training R²:       {dt_limited.score(X_train, y_train):.4f}")
print(f"Limited Tree Test R²:           {dt_limited.score(X_test, y_test):.4f}")

## 3. Random Forests (Bagging)

Instead of relying on one tree, a **Random Forest** builds hundreds of trees on different random subsets of data and averages their predictions. This "Wisdom of the Crowd" reduces variance and prevents overfitting.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def tune_model(model, param_grid, X_train, y_train):
    """
    Generic function to perform GridSearchCV.
    Note: For Trees, scaling is NOT technically required, but we include it in the pipeline
    for consistency with our Part 4 linear models.
    """
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    pipeline_param_grid = {f'model__{key}': value for key, value in param_grid.items()}

    grid_search = GridSearchCV(
        pipeline, pipeline_param_grid, cv=5, scoring='r2', n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    print(f"Best Parameters: {grid_search.best_params_}")
    print(f"Best CV R²: {grid_search.best_score_:.4f}")
    return grid_search.best_estimator_

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'max_features': [1.0, 'sqrt']
}

print("--- Tuning Random Forest ---")
best_rf = tune_model(RandomForestRegressor(random_state=42), rf_params, X_train, y_train)

## 4. Gradient Boosting

**Boosting** works differently. It builds trees sequentially. Each new tree focuses specifically on the errors (residuals) made by the previous trees. It's often the most accurate algorithm for tabular data.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

gb_params = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_iter': [100, 200, 300],
    'max_depth': [3, 5]
}

print("--- Tuning Gradient Boosting ---")
best_gb = tune_model(HistGradientBoostingRegressor(random_state=42), gb_params, X_train, y_train)

## 5. Feature Importance

Unlike coefficients in linear regression, tree models provide a "Feature Importance" score based on how much each feature contributed to reducing error across all trees.

In [ ]:
# Extract importance from the best Random Forest model
importances = best_rf.named_steps['model'].feature_importances_
feature_names = X.columns

feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
feat_imp.head(15).plot(kind='barh')
plt.title('Top 15 Most Important Features (Random Forest)')
plt.gca().invert_yaxis()
plt.show()

## 6. The Grand Finale: Linear vs. Non-Linear

Let's compare our best tree-based models against a quick optimized Lasso model (from Part 4) to see the final leaderboard.

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error

# 1. Quick optimized Lasso (re-running for context)
lasso_params = {'alpha': [0.0001, 0.001, 0.01]}
best_lasso = tune_model(Lasso(max_iter=5000), lasso_params, X_train, y_train)

models = {
    'Optimized Lasso (Linear)': best_lasso,
    'Optimized Random Forest': best_rf,
    'Optimized Gradient Boosting': best_gb
}

final_results = []
for name, model in models.items():
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    final_results.append({'Model': name, 'Test R²': f"{r2:.4f}", 'Test MSE': f"{mse:.4f}"})

pd.DataFrame(final_results)

## Conclusion

1.  **Complexity Wins**: In most cases, Gradient Boosting or Random Forest will outperform Lasso/Ridge because they can capture non-linear patterns (like `Overall Qual` being exponentially important).
2.  **Trade-offs**: Tree models are "black boxes"—they don't give us a simple equation like $y = mx + b$. We trade interpretability for predictive power.
3.  **No Silver Bullet**: On small datasets or data that is truly linear, OLS or Lasso can still win. However, for real-world tabular data like the Ames dataset, Ensemble methods are the gold standard.